This notebook will help us check the count of the incremental data present in some foreign table, based on the count set in the paramter row_count using taskValues, I am going to decide whether to run the subsequent ETL tasks/Pipline or not..

- Source = gcp_mysql_fc_wd36.logistics.shipments1
- Target (Silver) = catalog2_we47.schema2_we47.silver_shipmentsfc1
- **logic written below** -> "Count how many rows in source are newer than the latest row already loaded into silver"


**This is a classic incremental load pattern.**
- Example: -> Source Table: shipments1 (MySQL) ->(This is live operational data.)
| shipment_id | status     | updated_at          |
| ----------- | ---------- | ------------------- |
| 101         | Created    | 2026-02-18 10:00:00 |
| 102         | In Transit | 2026-02-19 09:30:00 |
| 103         | Delivered  | 2026-02-20 14:00:00 |
| 104         | Cancelled  | 2026-02-21 11:15:00 |
- Silver Table: silver_shipmentsfc1 -> You already loaded these 2 rows earlier
| shipment_id | status     | updated_at          |
| ----------- | ---------- | ------------------- |
| 101         | Created    | 2026-02-18 10:00:00 |
| 102         | In Transit | 2026-02-19 09:30:00 |

**-> Filter Source Using This Timestamp**
| shipment_id | updated_at          | > MAX = 2026-02-19 09:30? |
| ----------- | ------------------- | -------------------     |
| 101         | 2026-02-18 10:00:00 | ❌ No                   |
| 102         | 2026-02-19 09:30:00 | ❌ No (equal)           | 
| 103         | 2026-02-20 14:00:00 | ✅ Yes                  |
| 104         | 2026-02-21 11:15:00 | ✅ Yes                  |

**There are 2 new/updated shipments that are not yet in Silver**

In [0]:
#Incremental load needed
row_count = spark.sql("""
SELECT *
FROM gcp_mysql_fc_wd36.logistics.shipments1
WHERE updated_at > (
    SELECT COALESCE(MAX(updated_at), '1970-01-01')
    FROM catalog2_we47.schema2_we47.silver_shipmentsfc1
)
""").count()
print(row_count)

In [0]:
#This is a job.task level property to set a parameter/variable inside a task, which can be propogated/read by other tasks on the same lakeflow job
dbutils.jobs.taskValues.set(key="row_count", value=row_count)

#We use dbutils.jobs.taskValues to pass runtime metrics like row counts between dependent tasks, enabling conditional execution inside Lakeflow workflows